# v32.2 double-check notebook — independent S3 hotfix audit

Purpose: independently re-check the v32.2 claim without trusting any previous summary file.

This notebook:
1. Loads baseline v30.1 predictions and v27 predictions.
2. Loads `v32_1_b_advocate_candidates.json`.
3. Parses **inner generated text** for `Final Answer` and `VERDICT`, not only metadata.
4. Applies only the S3 repair for `idx=14` if all strict conditions pass.
5. Saves `v32_2_doublecheck_*` artifacts.
6. Reloads saved preds and recomputes all metrics/diffs.
7. Optionally compares with existing `v32_2_full_preds.json` if available.

This is a one-case artifact repair, not a broad validation of advocate generation.

In [ ]:
import os, re, json, math, copy
from pathlib import Path
from collections import Counter, defaultdict

LABELS = ['A','B','C','D','Yes','No','Unknown']
HOTFIX_IDX = 14
BANKED_DIFFS_VS_V27 = {71, 109, 125}
EXPECTED_DIFFS_VS_V30_1 = [14]
EXPECTED_DIFFS_VS_V27 = {14, 71, 109, 125}
BASELINE_V30_1_MACRO = 0.5934206145879246

CANDIDATE_DIRS = [
    Path('/kaggle/working'),
    Path('/kaggle/input'),
    Path('/kaggle/input/datasets/yixuanisthebest/v2333333'),
    Path('/mnt/data'),
    Path('.'),
]

def find_file(names, required=True):
    if isinstance(names, str):
        names = [names]
    for name in names:
        p = Path(name)
        if p.exists():
            return p
        for d in CANDIDATE_DIRS:
            q = d / name
            if q.exists():
                return q
    roots = [Path('/kaggle/working'), Path('/kaggle/input'), Path('/mnt/data'), Path('.')]
    for root in roots:
        if not root.exists():
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            depth = len(Path(dirpath).parts) - len(root.parts)
            if depth > 5:
                dirnames[:] = []
                continue
            for name in names:
                if name in filenames:
                    return Path(dirpath) / name
    if required:
        raise FileNotFoundError(f'Could not find any of: {names}')
    return None

def load_json(names, required=True):
    p = find_file(names, required=required)
    if p is None:
        return None, None
    with open(p, 'r', encoding='utf-8') as f:
        return json.load(f), p

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    return path

print('Ready. Search roots:', [str(p) for p in CANDIDATE_DIRS])

In [ ]:
# ============================================================
# Metric utilities — implemented here to avoid trusting summaries
# ============================================================

def row_idx(r):
    return int(r.get('idx'))

def pred_label(r):
    return r.get('pred') or r.get('prediction') or r.get('answer')

def gold_label(r):
    return r.get('gold') or r.get('label')

def is_mc_row(r):
    if 'is_mc' in r:
        return bool(r['is_mc'])
    g = gold_label(r)
    return g in ['A','B','C','D']

def prf_for_label(rows, label):
    tp = sum(1 for r in rows if gold_label(r)==label and pred_label(r)==label)
    fp = sum(1 for r in rows if gold_label(r)!=label and pred_label(r)==label)
    fn = sum(1 for r in rows if gold_label(r)==label and pred_label(r)!=label)
    precision = tp/(tp+fp) if tp+fp else 0.0
    recall = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    support = sum(1 for r in rows if gold_label(r)==label)
    pred_count = sum(1 for r in rows if pred_label(r)==label)
    return dict(precision=precision, recall=recall, f1=f1, support=support, pred_count=pred_count)

def compute_metrics(rows):
    n = len(rows)
    acc = sum(1 for r in rows if gold_label(r)==pred_label(r))/n
    per_label = {lab: prf_for_label(rows, lab) for lab in LABELS}
    macro_f1 = sum(per_label[lab]['f1'] for lab in LABELS)/len(LABELS)
    total_support = sum(per_label[lab]['support'] for lab in LABELS)
    weighted_f1 = sum(per_label[lab]['f1']*per_label[lab]['support'] for lab in LABELS)/total_support
    mc_rows = [r for r in rows if is_mc_row(r)]
    ynu_rows = [r for r in rows if not is_mc_row(r)]
    mc_labels = ['A','B','C','D']
    ynu_labels = ['Yes','No','Unknown']
    mc_per = {lab: prf_for_label(mc_rows, lab) for lab in mc_labels}
    ynu_per = {lab: prf_for_label(ynu_rows, lab) for lab in ynu_labels}
    mc_macro = sum(mc_per[lab]['f1'] for lab in mc_labels)/len(mc_labels)
    ynu_macro = sum(ynu_per[lab]['f1'] for lab in ynu_labels)/len(ynu_labels)
    return dict(
        n=n, acc=acc, macro_f1=macro_f1, weighted_f1=weighted_f1,
        mc_macro=mc_macro, ynu_macro=ynu_macro,
        per_label=per_label,
        gold_count=dict(Counter(gold_label(r) for r in rows)),
        pred_count=dict(Counter(pred_label(r) for r in rows)),
    )

def by_idx(rows):
    d = {row_idx(r): r for r in rows}
    assert len(d) == len(rows), 'Duplicate idx in rows'
    return d

def diffs_between(a_rows, b_rows):
    a = by_idx(a_rows); b = by_idx(b_rows)
    common = sorted(set(a) & set(b))
    return [i for i in common if pred_label(a[i]) != pred_label(b[i])]

def print_metrics(name, m):
    print(f"{name}: acc={m['acc']:.10f} macro={m['macro_f1']:.10f} weighted={m['weighted_f1']:.10f} MC={m['mc_macro']:.10f} YNU={m['ynu_macro']:.10f}")
    print('Pred count:', m['pred_count'])

In [ ]:
# ============================================================
# Load inputs
# ============================================================

v27_rows, v27_path = load_json('v27_standard_preds.json', required=True)
# Accept either v30_1_full_preds.json or a v32_1 rollback file as v30.1 baseline.
v30_rows, v30_path = load_json(['v30_1_full_preds.json', 'v30_full_preds.json', 'v32_1_full_preds.json'], required=True)
advocates, adv_path = load_json('v32_1_b_advocate_candidates.json', required=True)

assert isinstance(v27_rows, list) and isinstance(v30_rows, list)
assert len(v27_rows) == len(v30_rows) == 156, (len(v27_rows), len(v30_rows))

v27_m = compute_metrics(v27_rows)
v30_m = compute_metrics(v30_rows)
print('Loaded v27:', v27_path)
print('Loaded v30.1 baseline:', v30_path)
print('Loaded advocates:', adv_path)
print_metrics('v27', v27_m)
print_metrics('v30.1 baseline', v30_m)

assert abs(v30_m['macro_f1'] - BASELINE_V30_1_MACRO) < 1e-9, f"Unexpected v30.1 macro: {v30_m['macro_f1']}"
assert set(diffs_between(v27_rows, v30_rows)) == BANKED_DIFFS_VS_V27, diffs_between(v27_rows, v30_rows)
print('Baseline verified: v30.1 macro and banked diffs match.')

In [ ]:
# ============================================================
# Parse advocate text strictly — do not trust echo metadata alone
# ============================================================

FINAL_RE = re.compile(r'Final\s+Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', re.I)
VERDICT_RE = re.compile(r'VERDICT\s*[:\-]?\s*(VALID|INVALID)\b', re.I)

def norm_label(x):
    if x is None:
        return None
    s = str(x).strip()
    if s.upper() in ['A','B','C','D']:
        return s.upper()
    t = s.title()
    if t in ['Yes','No','Unknown']:
        return t
    return None

def parse_inner(adv):
    text = adv.get('text','') if isinstance(adv, dict) else str(adv)
    finals = FINAL_RE.findall(text)
    verdicts = VERDICT_RE.findall(text)
    inner_final = norm_label(finals[-1]) if finals else None
    inner_verdict = verdicts[-1].upper() if verdicts else None
    meta_final = norm_label(adv.get('final_answer')) if isinstance(adv, dict) else None
    meta_verdict = str(adv.get('verdict','')).upper() if isinstance(adv, dict) else None
    cited = bool(adv.get('cited')) if isinstance(adv, dict) else False
    return dict(inner_final=inner_final, inner_verdict=inner_verdict, meta_final=meta_final, meta_verdict=meta_verdict, cited=cited, text=text)

idx_key = str(HOTFIX_IDX)
assert idx_key in advocates, f'Missing advocate entry for idx {HOTFIX_IDX}'
entry = advocates[idx_key]
print('Advocate labels at idx14:', list(entry.keys()))
parsed = {lab: parse_inner(entry[lab]) for lab in ['Yes','No','Unknown']}
for lab, p in parsed.items():
    print('\n---', lab, '---')
    print('inner_final=', p['inner_final'], 'inner_verdict=', p['inner_verdict'], 'meta_final=', p['meta_final'], 'meta_verdict=', p['meta_verdict'], 'cited=', p['cited'])
    print(p['text'][:500].replace('\n',' ') + ('...' if len(p['text'])>500 else ''))

v30_by = by_idx(v30_rows)
assert pred_label(v30_by[HOTFIX_IDX]) == 'UNPARSEABLE', pred_label(v30_by[HOTFIX_IDX])
assert gold_label(v30_by[HOTFIX_IDX]) == 'Yes', gold_label(v30_by[HOTFIX_IDX])

valid_labels = [lab for lab, p in parsed.items() if p['inner_verdict'] == 'VALID']
print('\nValid advocate labels by inner verdict:', valid_labels)
assert valid_labels == ['Yes'], valid_labels
assert parsed['Yes']['inner_final'] == 'Yes'
assert parsed['Yes']['inner_verdict'] == 'VALID'
assert parsed['Yes']['cited'] is True
assert parsed['No']['inner_verdict'] == 'INVALID'
assert parsed['Unknown']['inner_verdict'] == 'INVALID'
print('S3 idx14 strict condition PASSED.')

In [ ]:
# ============================================================
# Apply the single repair and recompute metrics before saving
# ============================================================

new_rows = copy.deepcopy(v30_rows)
for r in new_rows:
    if row_idx(r) == HOTFIX_IDX:
        assert pred_label(r) == 'UNPARSEABLE'
        r['v32_2_old_pred'] = pred_label(r)
        r['pred'] = 'Yes'
        r['repair'] = 'v32_2_doublecheck_S3_idx14_UNPARSEABLE_to_Yes_inner_text_verified'
        r['source'] = r.get('source', '') + '|v32_2_doublecheck_S3'
        break
else:
    raise AssertionError('idx14 row not found')

new_m = compute_metrics(new_rows)
print_metrics('v32.2 doublecheck candidate', new_m)
print('Diffs vs v30.1:', diffs_between(v30_rows, new_rows))
print('Diffs vs v27:', diffs_between(v27_rows, new_rows))

assert diffs_between(v30_rows, new_rows) == EXPECTED_DIFFS_VS_V30_1
assert set(diffs_between(v27_rows, new_rows)) == EXPECTED_DIFFS_VS_V27
assert new_m['macro_f1'] > v30_m['macro_f1']
assert abs(new_m['mc_macro'] - v30_m['mc_macro']) < 1e-12
assert new_m['per_label']['Unknown']['f1'] >= v30_m['per_label']['Unknown']['f1'] - 1e-12
assert new_m['per_label']['Yes']['f1'] > v30_m['per_label']['Yes']['f1']
print('Pre-save metric/diff assertions PASSED.')

In [ ]:
# ============================================================
# Save, reload, recompute, and verify artifacts
# ============================================================

OUTDIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
std_preds_path = OUTDIR / 'v32_2_doublecheck_standard_preds.json'
std_summary_path = OUTDIR / 'v32_2_doublecheck_standard_summary.json'
full_preds_path = OUTDIR / 'v32_2_doublecheck_full_preds.json'
full_summary_path = OUTDIR / 'v32_2_doublecheck_full_summary.json'
risk_path = OUTDIR / 'v32_2_doublecheck_risk_report.json'

summary = dict(
    version='v32_2_doublecheck_s3_hotfix_idx14',
    selection_status='SELECT_V32_2_DOUBLECHECK',
    selected_source='v30.1 + independent S3 idx14 hotfix',
    rationale='Only flip idx14 because baseline is UNPARSEABLE and inner advocate text has unique VALID Yes; saved artifact reload verified.',
    baseline_v30_1={k:v for k,v in v30_m.items() if k not in ['per_label','gold_count','pred_count']},
    metrics=new_m,
    n_flips_from_v30_1=1,
    flipped_indices=EXPECTED_DIFFS_VS_V30_1,
    diffs_vs_v27=sorted(EXPECTED_DIFFS_VS_V27),
    strict_checks=dict(
        used_inner_text=True,
        yes_inner_final=parsed['Yes']['inner_final'],
        yes_inner_verdict=parsed['Yes']['inner_verdict'],
        no_inner_verdict=parsed['No']['inner_verdict'],
        unknown_inner_verdict=parsed['Unknown']['inner_verdict'],
        yes_cited=parsed['Yes']['cited'],
        mc_frozen=True,
        unknown_not_dropped=True,
    )
)

risk = dict(
    version='v32_2_doublecheck_risk_report',
    final_decision='SELECT_V32_2_DOUBLECHECK',
    metrics=new_m,
    delta_vs_v30_1={
        'acc': new_m['acc']-v30_m['acc'],
        'macro_f1': new_m['macro_f1']-v30_m['macro_f1'],
        'weighted_f1': new_m['weighted_f1']-v30_m['weighted_f1'],
        'mc_macro': new_m['mc_macro']-v30_m['mc_macro'],
        'ynu_macro': new_m['ynu_macro']-v30_m['ynu_macro'],
    },
    overfit_risk='MEDIUM_SINGLE_VAL_CASE_HOTFIX',
    underfit_risk='HIGH_REMAINING_YES_NO_ERRORS',
    artifact_risk='LOW_RELOADED_AND_RECOMPUTED',
    label_collapse=False,
    note='This is a one-case UNPARSEABLE repair, not a general advocate framework success.'
)

for p in [std_preds_path, full_preds_path]:
    save_json(new_rows, p)
for p in [std_summary_path, full_summary_path]:
    save_json(summary, p)
save_json(risk, risk_path)

for name, p in [('standard', std_preds_path), ('full', full_preds_path)]:
    rows_reloaded, _ = load_json(str(p), required=True)
    m_reloaded = compute_metrics(rows_reloaded)
    assert abs(m_reloaded['macro_f1'] - new_m['macro_f1']) < 1e-12, (name, m_reloaded['macro_f1'], new_m['macro_f1'])
    assert diffs_between(v30_rows, rows_reloaded) == EXPECTED_DIFFS_VS_V30_1, (name, diffs_between(v30_rows, rows_reloaded))
    assert set(diffs_between(v27_rows, rows_reloaded)) == EXPECTED_DIFFS_VS_V27, (name, diffs_between(v27_rows, rows_reloaded))
    assert abs(m_reloaded['mc_macro'] - v30_m['mc_macro']) < 1e-12
    print_metrics(f'Reloaded {name}', m_reloaded)

print('\nALL DOUBLECHECK ASSERTS PASSED')
print('Wrote:')
for p in [std_preds_path, std_summary_path, full_preds_path, full_summary_path, risk_path]:
    print(' -', p)

In [ ]:
# ============================================================
# Optional comparison with Claude's v32_2_full_preds.json if present
# ============================================================

existing_path = find_file('v32_2_full_preds.json', required=False)
if existing_path is None:
    print('No existing v32_2_full_preds.json found; skipping cross-file comparison.')
else:
    existing_rows, _ = load_json(str(existing_path), required=True)
    existing_m = compute_metrics(existing_rows)
    print('Found existing v32_2_full_preds.json:', existing_path)
    print_metrics('Existing v32_2_full_preds', existing_m)
    assert diffs_between(existing_rows, new_rows) == [], 'Doublecheck preds differ from existing v32_2_full_preds.json'
    print('Existing v32_2_full_preds.json exactly matches doublecheck preds.')

## Interpretation

If all cells pass, the v32.2 hotfix is independently verified:

- Best becomes `v32.2_doublecheck` / equivalent to `v32_2_full_*`.
- The only new change from v30.1 is `idx 14: UNPARSEABLE -> Yes`.
- This is a **surgical artifact/parse repair**, not a broad success of the v32 advocate framework.
- From v33 onward, use proof-grounded rules or stricter validation; do not relax gates for normal Yes/No flips.